# Notebook 00 — Project Setup & Dataset Verification
**Run this first before anything else.**

This notebook:
1. Shows exactly where to place your HAM10000 files
2. Creates the correct folder structure
3. Verifies your dataset is placed correctly
4. Tells you what's missing if anything is wrong

---
### Where to put your files

```
robust_medical_vision/
└── dataset/
    ├── HAM10000_metadata.csv     ← your CSV here
    └── images/                   ← ALL .jpg images here (both parts merged)
```

### Merging both image folders (run in Terminal first)
```bash
mkdir -p dataset/images
cp /path/to/HAM10000_images_part1/*.jpg dataset/images/
cp /path/to/HAM10000_images_part2/*.jpg dataset/images/
```


In [2]:
import os, sys
# Make sure we can import from the project root
sys.path.insert(0, os.path.abspath('..'))
print("Python path set to project root ✅")
print(f"Working directory: {os.getcwd()}")


Python path set to project root ✅
Working directory: /Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/notebooks


## Step 1 — Create folder structure

In [3]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
DATASET_DIR  = PROJECT_ROOT / 'dataset'
IMAGES_DIR   = DATASET_DIR  / 'images'
OUTPUTS_DIR  = PROJECT_ROOT / 'outputs'

for folder in [DATASET_DIR, IMAGES_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"  ✅ {folder.relative_to(PROJECT_ROOT)}/")

print("\nFolder structure ready.")


  ✅ dataset/
  ✅ dataset/images/
  ✅ outputs/

Folder structure ready.


## Step 2 — Check metadata CSV

In [4]:
import csv
from collections import Counter

metadata_path = PROJECT_ROOT / 'dataset' / 'HAM10000_metadata.csv'

if not metadata_path.exists():
    print(f"❌ NOT FOUND: {metadata_path}")
    print("   → Place your HAM10000_metadata.csv inside dataset/")
else:
    with open(metadata_path) as f:
        reader = csv.DictReader(f)
        rows   = list(reader)
        cols   = reader.fieldnames

    print(f"✅ Found metadata CSV")
    print(f"   Rows:    {len(rows):,}")
    print(f"   Columns: {cols}")
    print(f"   Unique lesions: {len(set(r['lesion_id'] for r in rows)):,}")

    dx_counts = Counter(r['dx'] for r in rows)
    print(f"\nClass distribution:")
    for cls, count in sorted(dx_counts.items(), key=lambda x: -x[1]):
        bar = '█' * int(count / 100)
        print(f"  {cls:6s}: {count:5d}  {bar}")


✅ Found metadata CSV
   Rows:    10,015
   Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']
   Unique lesions: 7,470

Class distribution:
  nv    :  6705  ███████████████████████████████████████████████████████████████████
  mel   :  1113  ███████████
  bkl   :  1099  ██████████
  bcc   :   514  █████
  akiec :   327  ███
  vasc  :   142  █
  df    :   115  █


## Step 3 — Check images folder

In [5]:
jpgs = list(IMAGES_DIR.glob('*.jpg'))
print(f"Images folder: {IMAGES_DIR}")
print(f"Found: {len(jpgs):,} .jpg files")

if len(jpgs) == 0:
    print("\n❌ No images found.")
    print("   → Copy all .jpg files from both HAM10000_images_part1 and _part2 into dataset/images/")
elif len(jpgs) < 5000:
    print(f"\n⚠️  Only {len(jpgs)} images — expected ~10,015")
    print("   → Did you copy BOTH part1 AND part2?")
else:
    print(f"\n✅ Images look complete (expected ~10,015)")
    print(f"   First 3 files: {[f.name for f in jpgs[:3]]}")


Images folder: /Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/dataset/images
Found: 10,015 .jpg files

✅ Images look complete (expected ~10,015)
   First 3 files: ['ISIC_0030858.jpg', 'ISIC_0030680.jpg', 'ISIC_0033389.jpg']


## Step 4 — Cross-check CSV image IDs vs disk

In [6]:
with open(metadata_path) as f:
    image_ids = [row['image_id'] for row in csv.DictReader(f)]

on_disk  = {f.stem for f in IMAGES_DIR.glob('*.jpg')}
missing  = [i for i in image_ids if i not in on_disk]

print(f"Images listed in CSV:   {len(image_ids):,}")
print(f"Images found on disk:   {len(on_disk):,}")
print(f"Missing from disk:      {len(missing)}")

if len(missing) == 0:
    print("\n✅ All images accounted for — you're ready to proceed!")
    print("\n→ NEXT: Open notebook 01_eda.ipynb")
else:
    print(f"\n⚠️  {len(missing)} images in CSV not found on disk.")
    print(f"   First few missing: {missing[:5]}")


Images listed in CSV:   10,015
Images found on disk:   10,015
Missing from disk:      0

✅ All images accounted for — you're ready to proceed!

→ NEXT: Open notebook 01_eda.ipynb
